<a href="https://colab.research.google.com/github/RaihanArkan4/050825/blob/main/Kelompok_kka.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd
import numpy as np

# --- 1. DATA LOADING & INSPECTION ---
print("--- DATA LOADING & INSPECTION ---")
# Membaca data dari file Excel
df = pd.read_excel('dataset_sensor_iot_kelas.xlsx')

print("\n5 Baris Pertama:")
print(df.head())
print("\nInfo Dataset:")
print(df.info())
print("\nStatistik Ringkas:")
print(df.describe())
print("\nUkuran Data (Baris, Kolom):")
print(df.shape)

# --- 2. DATA CLEANING ---
print("\n--- DATA CLEANING ---")
print("Jumlah data kosong sebelum dibersihkan:\n", df.isnull().sum())
print("Jumlah duplikat sebelum dihapus:", df.duplicated().sum())

# a. Menghapus baris duplikat
df = df.drop_duplicates()

# b. Menangani Missing Value
# Mengisi kelembapan kosong dengan rata-rata kelembapan agar data tetap utuh
rata_kelembapan = df['kelembapan_persen'].mean()
df['kelembapan_persen'] = df['kelembapan_persen'].fillna(rata_kelembapan)

# Menghapus sisa baris yang kosong pada 'suhu_celsius' atau 'status_sensor' (karena ini data krusial)
df = df.dropna(subset=['suhu_celsius', 'status_sensor'])

# c. Menyeragamkan penulisan teks
# Mengubah semua nama ruang dan status menjadi huruf kapital (Misal: 'Aktif' dan 'nonaktif' menjadi 'AKTIF' / 'NONAKTIF')
df['ruang'] = df['ruang'].str.upper()
df['status_sensor'] = df['status_sensor'].str.upper()

# d. Memperbaiki Tipe Data Suhu (Trick Khusus Excel)
# Fungsi ini mengembalikan format tanggal Excel (misal 29 Jan) kembali menjadi angka desimal (29.1)
def perbaiki_suhu(x):
    if isinstance(x, pd.Timestamp) or type(x).__name__ == 'datetime':
        return float(f"{x.day}.{x.month}")
    return x

# Terapkan fungsi perbaikan tanggal, lalu hapus teks 'C' atau 'derajat' pada kolom suhu
df['suhu_celsius'] = df['suhu_celsius'].apply(perbaiki_suhu)
df['suhu_celsius'] = df['suhu_celsius'].astype(str).str.replace('C', '', case=False, regex=False).str.replace('derajat', '', case=False, regex=False).str.strip()

# Terakhir, pastikan kolom suhu terbaca sebagai tipe data desimal (float)
df['suhu_celsius'] = pd.to_numeric(df['suhu_celsius'], errors='coerce')

# --- 3. DATA MANIPULATION ---
print("\n--- DATA MANIPULATION ---")
# a. Filtering (Hanya menganalisis data saat sensor sedang AKTIF)
sensor_aktif = df[df['status_sensor'] == 'AKTIF']

# b. Sorting (Mengurutkan ruangan dari suhu yang paling panas ke dingin)
urut_suhu = df.sort_values(by='suhu_celsius', ascending=False)

# c. Kolom Turunan (Membuat kolom baru: konversi suhu Celcius ke Fahrenheit)
df['suhu_fahrenheit'] = (df['suhu_celsius'] * 9/5) + 32

# d. Groupby / Agregasi (Melihat rata-rata suhu di setiap ruangan)
ringkasan_suhu = df.groupby('ruang')['suhu_celsius'].mean().reset_index()
# Mengurutkan hasil rata-rata
ringkasan_suhu = ringkasan_suhu.sort_values(by='suhu_celsius', ascending=False)
print("\nRata-rata Suhu per Ruangan:\n", ringkasan_suhu)

# --- 4. MENYIMPAN DATASET BERSIH ---
df.to_csv('dataset_iot_bersih.csv', index=False)
print("\nData berhasil disimpan sebagai 'dataset_iot_bersih.csv' untuk tahap visualisasi!")

--- DATA LOADING & INSPECTION ---

5 Baris Pertama:
  id_bacaan                     waktu                 ruang  \
0   SNS0018  4 Agustus 2026 jam 13:00        LAB-KOMPUTER-1   
1   SNS0051       2026-08-08 07:00:00        kelas xi rpl 3   
2   SNS0030          05/08/2026 15.00              XI-RPL-3   
3   SNS0015       2026-08-04 11:00:00  Kelas Lab Komputer 2   
4   SNS0038       2026-08-06 13:00:00        Kelas XI RPL 1   

          suhu_celsius  kelembapan_persen status_sensor  
0  2026-01-29 00:00:00                NaN      Nonaktif  
1  2026-08-24 00:00:00               64.0         Aktif  
2  2026-03-30 00:00:00               45.0         AKTIF  
3         26.6 derajat               62.0      nonaktif  
4                27.0C               59.0      nonaktif  

Info Dataset:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 74 entries, 0 to 73
Data columns (total 6 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   id_